# InternVL2-8B × {Uniform-32, AKS+E-32, KTV+E-32} trên UrbanVideo-Bench

Notebook điều khiển. Phần nặng chạy trên Modal, notebook chỉ phát lệnh, tải kết quả và vẽ biểu đồ.

| Ô | Việc | Chi phí ước tính |
|---|---|---|
| 0 | Cấu hình, hàm tiện ích | — |
| 1 | Tạo/kiểm tra subset (dev_300, subset_1000) | 0 |
| 2 | Chuẩn bị môi trường trên tài khoản Modal mới (1 lần) | ~$0.5 |
| 3 | Chạy thử 20 câu (**chờ xem kết quả**) | ~$1 |
| 4 | Phân tích t1 trên cache CLIP (local) | 0 |
| 5 | **Phát lệnh chạy qua đêm** (`--detach`) | ~$8–10 |
| 6 | Xem tiến độ, tải kết quả, chấm điểm, vẽ biểu đồ | 0 |
| 7 | (Sau) chạy full 5.355 câu | ~$15–20 |

Chạy với kernel `.venv` của dự án. Ô 0 tự chuyển thư mục làm việc về gốc dự án.

In [1]:
# Ô 0 — Cấu hình + tiện ích
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

PROFILE = "duonghoangminh15112005"      # tài khoản Modal mới
MODEL = "OpenGVLab/InternVL2-8B"
MODEL_SHORT = MODEL.split("/")[-1]
VOL = "urbanvideo-vol"
T1S = (0.8, 0.3, 0.2)                    # phải khớp pipeline.T1S
ENV = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"}
MODAL = [sys.executable, "-m", "modal"]


def sh(*args, check=True):
    """Chạy một lệnh modal, chờ xong, in phần cuối output."""
    cmd = [*MODAL, *args]
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, env=ENV, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(p.stdout[-6000:], p.stderr[-3000:])
    if check and p.returncode:
        raise RuntimeError(f"exit {p.returncode}")
    return p


def launch_detached(name, *args):
    """Phát `modal run --detach ...` chạy nền, log ra outputs/logs/<name>.log.
    Đóng notebook / ngủ máy vẫn không sao: app trên Modal chạy tiếp, kết quả lưu trên Volume."""
    os.makedirs("outputs/logs", exist_ok=True)
    log = open(f"outputs/logs/{name}.log", "w", encoding="utf-8")
    flags = (subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS) if os.name == "nt" else 0
    p = subprocess.Popen([*MODAL, "run", "--detach", *args], env=ENV, stdout=log,
                         stderr=subprocess.STDOUT, creationflags=flags)
    print(f"Đã phát lệnh (pid {p.pid}). Log: outputs/logs/{name}.log")
    return p


def tail(name, n=30):
    with open(f"outputs/logs/{name}.log", encoding="utf-8", errors="replace") as f:
        print("".join(f.readlines()[-n:]))

In [2]:
# Ô 1 — Subset: dev_300 (dò t1) và subset_1000 (báo cáo), phân tầng theo 16 task, seed 0
from subset import load_qids

mcq = pd.read_parquet("outputs/data/MCQ.parquet")
mcq["Question_id"] = mcq["Question_id"].astype(str)
if not (Path("outputs/subsets/subset_1000.txt").exists() and Path("outputs/subsets/dev_300.txt").exists()):
    subprocess.run([sys.executable, "subset.py"], env=ENV, check=True)   # cố định theo seed

sub = {n: load_qids(f"outputs/subsets/{n}.txt") for n in ("dev_300", "subset_1000")}
assert not set(sub["dev_300"]) & set(sub["subset_1000"]), "dev_300 và subset_1000 bị trùng!"
tbl = pd.DataFrame({"full": mcq["question_category"].value_counts()})
for n, q in sub.items():
    tbl[n] = mcq[mcq["Question_id"].isin(q)]["question_category"].value_counts()
tbl["full_%"] = (tbl["full"] / tbl["full"].sum() * 100).round(1)
tbl["subset_1000_%"] = (tbl["subset_1000"] / tbl["subset_1000"].sum() * 100).round(1)
print({n: (len(q), mcq[mcq.Question_id.isin(q)].video_id.nunique()) for n, q in sub.items()}, "(câu, video)")
tbl

{'dev_300': (300, 263), 'subset_1000': (1000, 716)} (câu, video)


,full,dev_300,subset_1000,full_%,subset_1000_%
question_category,,,,,
Action Generation,1183,66,221,22.1,22.1
Landmark Position,876,49,164,16.4,16.4
Progress Evaluation,781,44,146,14.6,14.6
Trajectory Captioning,421,24,79,7.9,7.9
Goal Detection,320,18,60,6.0,6.0
Cognitive Map,270,15,50,5.0,5.0
High-level Planning,261,15,49,4.9,4.9
Association Reasoning,237,13,44,4.4,4.4
Start/End Position,146,8,27,2.7,2.7


In [3]:
# Ô 2 — Môi trường trên tài khoản mới (chạy 1 lần)
p = sh("profile", "current")
assert PROFILE in p.stdout, f"Profile active không phải {PROFILE}: chạy `modal profile activate {PROFILE}`"

for v in (VOL, "hf-cache"):
    sh("volume", "create", v, check=False)          # đã có thì báo lỗi, bỏ qua

# Dataset + InternVL2-8B + CLIP (AKS) + DINOv2-large (KTV). Mất ~10 phút.
sh("run", "download.py", "--vlm", MODEL, "--scorers", "clip,dinov2")

# Dùng lại dữ liệu đã tính ở tài khoản cũ: điểm CLIP (AKS chọn frame gần như miễn phí)
# và uniform_32f (không phụ thuộc mô hình).
sh("volume", "put", VOL, "outputs/cache/clip", "/cache/aks/clip", "--force")
sh("volume", "put", VOL, "outputs/selections/uniform_32f.json", "/selections/uniform_32f.json", "--force")
sh("volume", "ls", VOL, "/")

$ d:\Work\Learn\HK7_2026-2027\Data_processing\Final_exam\.venv\Scripts\python.exe -m modal profile current
duonghoangminh15112005
 
$ d:\Work\Learn\HK7_2026-2027\Data_processing\Final_exam\.venv\Scripts\python.exe -m modal volume create urbanvideo-vol
 ┌─ Error ─────────────────────────────────────────────────────────────────────┐
│ Volume 'urbanvideo-vol' already exists in environment 'main'                │
└─────────────────────────────────────────────────────────────────────────────┘

$ d:\Work\Learn\HK7_2026-2027\Data_processing\Final_exam\.venv\Scripts\python.exe -m modal volume create hf-cache
 ┌─ Error ─────────────────────────────────────────────────────────────────────┐
│ Volume 'hf-cache' already exists in environment 'main'                      │
└─────────────────────────────────────────────────────────────────────────────┘

$ d:\Work\Learn\HK7_2026-2027\Data_processing\Final_exam\.venv\Scripts\python.exe -m modal run download.py --vlm OpenGVLab/InternVL2-8B --scorers clip

CompletedProcess(args=['d:\\Work\\Learn\\HK7_2026-2027\\Data_processing\\Final_exam\\.venv\\Scripts\\python.exe', '-m', 'modal', 'volume', 'ls', 'urbanvideo-vol', '/'], returncode=0, stdout='data\nstatus\ncache\nselections\n', stderr='')

### Ô 3 — Chạy thử 20 câu (không detach, chờ xong)
Chỉ sang ô 5 khi kết quả in ra có `"ok": true`: mọi câu có câu trả lời, `unparsed = 0`, đủ ~32 frame,
engine khởi động không lỗi. Sau đó `modal app list` không được còn app nào đang chạy sót.

In [4]:
p = sh("run", "pipeline.py", "--stage", "smoke", "--limit", "20", check=False)
sh("app", "list")

$ d:\Work\Learn\HK7_2026-2027\Data_processing\Final_exam\.venv\Scripts\python.exe -m modal run pipeline.py --stage smoke --limit 20
ine 1232, in mapper_func_wrapper
    return (tup[0], await async_mapper_func(tup[1]))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pkg/modal/parallel_map.py", line 915, in fetch_output
    raise e
  File "/pkg/modal/parallel_map.py", line 910, in fetch_output
    output = await _process_result(item.result, item.data_format, input_plane_stub, client)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pkg/modal/_utils/function_utils.py", line 574, in _process_result
    raise RemoteError(result.exception)
modal.exception.RemoteError
  File "/pkg/modal/_runtime/container_io_manager.py", line 987, in handle_input_exception
    yield
  File "/pkg/modal/_container_entrypoint.py", line 189, in run_input_sync
    values = io_context.call_function_sync()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 

CompletedProcess(args=['d:\\Work\\Learn\\HK7_2026-2027\\Data_processing\\Final_exam\\.venv\\Scripts\\python.exe', '-m', 'modal', 'app', 'list'], returncode=0, stdout='                                     Apps                                      \n                                                                               \n                              Descript                     Created    Stopped  \n  App ID                      ion        State     Tasks   at         at       \n ───────────────────────────────────────────────────────────────────────────── \n  ap-d6Muph51YpB6nDuo22WVR8   uvb-pipe   stopped   3       2026-11-   2026-11  \n                              line                         19 00:41   -19      \n                                                           SE Asia    00:48    \n                                                           Standard   SE Asia  \n                                                           Time       Standar  \n                       

In [ ]:
# Ô 4 — Phân tích t1 trên cache CLIP (local, không tốn GPU)
# t1 càng thấp, ADA càng dừng sớm, frame càng gom cụm quanh chỗ khớp câu hỏi
from methods.aks.ada import _judge_and_split, ada_select

files = glob.glob("outputs/cache/clip/*.json")
rows = []
for t1 in sorted({*T1S, 0.5, 0.4, 0.1}, reverse=True):
    d0, d5, gap = [], [], []
    for f in files:
        with open(f) as fh:
            c = json.load(fh)
        sc, fr = np.asarray(c["scores"], float), c["frames"]
        if len(sc) <= 32:
            continue
        n = (sc - sc.min()) / (np.ptp(sc) or 1)
        segs = _judge_and_split([dict(score=n, fn=list(range(len(n))), depth=0)], 32, t1, -100, 5)
        ds = [s["depth"] for s in segs]
        d0.append(min(ds) == 0)
        d5.append(np.mean([x == 5 for x in ds]))
        sel = np.array(ada_select(sc, fr, 32, t1))
        gap.append(np.diff(np.r_[fr[0], sel, fr[-1]]).max() / ((fr[-1] - fr[0]) / 31))
    rows.append({"t1": t1, "% câu dừng ở độ sâu 0": 100 * np.mean(d0),
                 "% đoạn chia tới độ sâu 5": 100 * np.mean(d5),
                 "khoảng trống max / bước uniform": np.mean(gap)})
pd.DataFrame(rows).round(2)

### Ô 5 — Chạy qua đêm (detach)
Stage `night` (xem `pipeline.py`):
1. Chọn frame cho dev_300 ∪ subset_1000: Uniform, KTV+E, AKS+E × t1 ∈ {0.8, 0.3, 0.2}
2. Trả lời dev_300 cho AKS+E × 3 giá trị t1, rồi **chọn t1 theo quy tắc ghi trước**
   (Overall trên dev_300 cao nhất, hòa thì lấy t1 lớn hơn)
3. Trả lời subset_1000: Uniform-32, AKS+E-32 (t1 đã chọn), KTV+E-32

Ô này deploy app rồi gửi lệnh chạy và thoát ngay. Thấy dòng `Đã gửi stage 'night'` là có thể tắt máy.
**Mở lại máy:** luôn chạy ô 0 trước (kernel mới không còn biến cũ), rồi chạy ô 6.

In [ ]:
# deploy app rồi spawn run_stage: chạy độc lập với máy local (không dùng modal run --detach,
# vì tắt tiến trình local sẽ gửi tín hiệu hủy, xem RUN_LOG 2026-11-19)
p = subprocess.run([sys.executable, "launch.py", "night"], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:], p.stderr[-2000:])

In [ ]:
# Xem tiến độ bất cứ lúc nào (kể cả sau khi mở lại máy): chạy ô 0 rồi ô 6

In [2]:
# Ô 6 — Tiến độ (chạy lại bao nhiêu lần cũng được)
os.makedirs("outputs/status", exist_ok=True)
sh("volume", "get", VOL, "status/night.json", "outputs/status/night.json", "--force", check=False)
if Path("outputs/status/night.json").exists():
    with open("outputs/status/night.json", encoding="utf-8") as f:
        st = json.load(f)
    print("done:", st["done"], "| chosen_t1:", st.get("chosen_t1"), "| error:", st.get("error"))
    display(pd.DataFrame(st["steps"]))

NameError: name 'os' is not defined

In [1]:
# Tải kết quả rồi chấm điểm (evaluate.py ghi outputs/summary.csv, outputs/pairwise.csv)
sh("volume", "get", VOL, "results", "outputs/", "--force")
shutil.rmtree("outputs/results/parts", ignore_errors=True)   # file từng shard, evaluate không cần
files = sorted(glob.glob(f"outputs/results/*{MODEL_SHORT}*.csv"))
p = subprocess.run([sys.executable, "evaluate.py", *files], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-8000:], p.stderr[-2000:])

NameError: name 'sh' is not defined

In [ ]:
import matplotlib.pyplot as plt

from config import TASK_GROUPS

TASKS = [t for ts in TASK_GROUPS.values() for t in ts]
PAPER_UNIFORM = 28.1   # InternVL2-8B[32f], Table 3 của UrbanVideo-Bench (Avg = micro)

s = pd.read_csv("outputs/summary.csv")
s = s[(s.model == MODEL_SHORT) & (s.subset == "subset_1000")].reset_index(drop=True)
display(s[["tag", "n", "unparsed", "frames", "Overall", "CI_lo", "CI_hi", "Macro", *TASK_GROUPS]].round(1))
u = s[s.tag == "uniform_32f"].Overall
if len(u):
    print(f"Tái hiện baseline: Uniform-32 = {u.iloc[0]:.1f} so với paper {PAPER_UNIFORM} "
          f"-> {'ĐẠT' if abs(u.iloc[0] - PAPER_UNIFORM) <= 3 else 'KHÔNG ĐẠT'} (ngưỡng ±3)")
if Path("outputs/pairwise.csv").exists():
    pw = pd.read_csv("outputs/pairwise.csv")
    display(pw[(pw.model == MODEL_SHORT) & (pw.subset == "subset_1000")].round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 3]})
x = np.arange(len(s))
ax[0].bar(x, s.Overall, yerr=[s.Overall - s.CI_lo, s.CI_hi - s.Overall], capsize=4, color="#4C78A8")
ax[0].axhline(PAPER_UNIFORM, ls="--", c="gray", label="paper Uniform-32")
ax[0].legend()
ax[0].set_xticks(x, s.tag, rotation=20)
ax[0].set_ylabel("Overall (%)")
ax[0].set_title("Overall, CI 95%")
tasks = [t for t in TASKS if t in s]
w = 0.8 / max(len(s), 1)
for i, r in s.iterrows():
    ax[1].bar(np.arange(len(tasks)) + i * w, r[tasks].astype(float), w, label=r.tag)
ax[1].set_xticks(np.arange(len(tasks)) + 0.4 - w / 2, tasks, rotation=60, ha="right")
ax[1].set_title("Theo task")
ax[1].legend()
plt.tight_layout()
plt.show()

### Ô 7 — (Sau) chạy full 5.355 câu
Chỉ chạy khi lượt `night` đã xong, baseline tái hiện đạt, và còn đủ credit (~$15–20).
t1 lấy từ `status/night.json`.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    p = subprocess.run([sys.executable, "launch.py", "full"], env=ENV, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    print(p.stdout[-3000:], p.stderr[-2000:])

## Ô 8 — Ngân sách 8 frame (stage `f8`)

Cùng quy trình như `night` nhưng chỉ **8 frame/câu** (khoảng 11 giây/frame với video trung vị). Ở mức ít frame,
chọn frame thông minh dễ tạo khác biệt hơn so với 32 frame.

| | Uniform-8 | AKS+E-8 | KTV+E-8 |
|---|---|---|---|
| Cách chọn | 8 frame lấy đều | CLIP + ADA, **all_depth = 3** (ADA lấy int(8/2^depth) frame/đoạn; all_depth = 5 cho 0 frame), t1 dò lại trên dev_300 ∈ {0.8, 0.3, 0.2} | KMeans 8 cụm trên DINOv2 |
| Frame đầu/cuối | có sẵn | ép thêm (+E) | ép thêm (+E) |
| Tập ứng viên | — | như lượt 32 frame (1 fps, tối thiểu 64) | như lượt 32 frame (`cand_k = 32`) |

Dùng lại cache điểm CLIP và feature DINOv2 nên chọn frame gần như miễn phí. Chi phí ước tính ~$2–4 (8 frame ít token hơn 4 lần).

In [ ]:
# Ô 8a — Phát lệnh f8 (deploy + spawn, chạy độc lập với máy local). Chỉ chạy 1 lần!
p = subprocess.run([sys.executable, "launch.py", "f8"], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:], p.stderr[-2000:])

In [ ]:
# Ô 8b — Tiến độ f8 (chạy lại bao nhiêu lần cũng được)
sh("volume", "get", VOL, "status/f8.json", "outputs/status/f8.json", "--force", check=False)
if Path("outputs/status/f8.json").exists():
    with open("outputs/status/f8.json", encoding="utf-8") as f:
        st8 = json.load(f)
    print("done:", st8["done"], "| chosen_t1:", st8.get("chosen_t1"), "| error:", st8.get("error"))
    display(pd.DataFrame(st8["steps"]))

In [ ]:
# Ô 8c — Tải kết quả, chấm điểm và so sánh 8 frame với 32 frame
sh("volume", "get", VOL, "results", "outputs/", "--force")
shutil.rmtree("outputs/results/parts", ignore_errors=True)
p = subprocess.run([sys.executable, "evaluate.py"], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stderr[-2000:])

import matplotlib.pyplot as plt

s = pd.read_csv("outputs/summary.csv")
s = s[(s.model == MODEL_SHORT) & (s.subset == "subset_1000")].copy()
s["nf"] = s.tag.str.extract(r"_(\d+)f$").astype(int)
s["method"] = s.tag.str.extract(r"^(uniform|aks|ktv)")
cols = ["method", "nf", "tag", "Overall", "CI_lo", "CI_hi", "Macro", "Recall", "Perception", "Reasoning",
        "Navigation", "src:AerialVLN", "src:EmbodiedCity", "src:RealWorld"]
display(s.sort_values(["nf", "method"])[cols].round(1))

pw = pd.read_csv("outputs/pairwise.csv")
pw = pw[(pw.model == MODEL_SHORT) & (pw.subset == "subset_1000")]
pw = pw[pw.A.str.extract(r"_(\d+)f$")[0] == pw.B.str.extract(r"_(\d+)f$")[0]]   # chỉ so cùng số frame
display(pw[["A", "B", "acc_A", "acc_B", "diff", "diff_lo", "diff_hi", "only_A", "only_B", "p", "p_holm"]].round(3))

fig, ax = plt.subplots(figsize=(7, 4))
for m, c in (("uniform", "#888888"), ("aks", "#E45756"), ("ktv", "#4C78A8")):
    g = s[s.method == m].sort_values("nf")
    ax.errorbar(g.nf, g.Overall, yerr=[g.Overall - g.CI_lo, g.CI_hi - g.Overall], marker="o",
                capsize=4, label=m, color=c)
ax.axhline(28.1, ls="--", c="gray", lw=1, label="paper Uniform-32")
ax.set_xticks([8, 32]); ax.set_xlabel("số frame"); ax.set_ylabel("Overall (%)")
ax.set_title("InternVL2-8B, subset_1000"); ax.legend()
plt.tight_layout(); plt.show()

## Ô 9 — Baseline "mù" (0 frame, stage `blind`)

InternVL2-8B nhận **đúng prompt + câu hỏi như mọi lượt khác nhưng không có frame nào**. Mục đích: đo xem mô hình
thực sự dựa vào video bao nhiêu.
- Nếu Overall của 0 frame ≈ Uniform-8/32 → mô hình chủ yếu đoán từ câu hỏi, chọn frame khó tạo khác biệt.
- Nếu thấp hơn rõ → video có đóng góp, phần chênh là "trần" mà chọn frame có thể khai thác.

Chạy trên subset_1000, chi phí ~$1. Ngẫu nhiên (Table 3 của paper) = 19,7.

In [ ]:
# Ô 9a — Phát lệnh blind (deploy + spawn). Chỉ chạy 1 lần!
p = subprocess.run([sys.executable, "launch.py", "blind"], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:], p.stderr[-2000:])

In [ ]:
# Ô 9b — Tiến độ + so sánh 0 frame với 8 / 32 frame
sh("volume", "get", VOL, "status/blind.json", "outputs/status/blind.json", "--force", check=False)
with open("outputs/status/blind.json", encoding="utf-8") as f:
    stb = json.load(f)
print("done:", stb["done"], "| error:", stb.get("error"))
display(pd.DataFrame(stb["steps"]))

if stb["done"]:
    from evaluate import extract_option
    name = f"textonly_0f__{MODEL_SHORT}__subset_1000.csv"
    sh("volume", "get", VOL, f"results/{name}", f"outputs/results/{name}", "--force")
    subprocess.run([sys.executable, "evaluate.py"], env=ENV, capture_output=True)

    s = pd.read_csv("outputs/summary.csv")
    s = s[(s.model == MODEL_SHORT) & (s.subset == "subset_1000")]
    show = ["textonly_0f", "uniform_8f", "uniform_32f", "ktv-e_8f", "ktv-e_32f"]
    display(s.set_index("tag").loc[[t for t in show if t in set(s.tag)],
            ["Overall", "CI_lo", "CI_hi", "Macro", "Recall", "Perception", "Reasoning", "Navigation"]].round(1))

    P = {}
    for t in show:
        d = pd.read_csv(f"outputs/results/{t}__{MODEL_SHORT}__subset_1000.csv").sort_values("Question_id")
        P[t] = d["Output"].map(extract_option).to_numpy()
    print("Tỉ lệ câu trả lời GIỐNG HỆT với baseline 0 frame:")
    for t in show[1:]:
        print(f"  {t:12s}: {100 * (P[t] == P['textonly_0f']).mean():.1f}%")

## Ô 10 — Baseline mù cho Qwen2.5-VL-7B: mô hình nào dựa vào video nhiều hơn?

Chạy 0 frame cho **Qwen2.5-VL-7B** trên subset_1000 (~$1) rồi so với kết quả **có video** đã có sẵn từ tháng 10
(Uniform-32 và AKS-CLIP-32 trên full 5.355 câu, H100 batch 32 — cùng GPU/batch với lượt 0 frame), lọc đúng 1.000 câu của subset.

Nếu Qwen dựa vào video nhiều hơn InternVL2 (khoảng cách 0 frame → 32 frame lớn hơn ~3,5 điểm), Qwen là mô hình
phù hợp hơn để thử KTV.

In [ ]:
# Ô 10a — Tải Qwen2.5-VL-7B lên hf-cache (1 lần) rồi phát lệnh blind cho Qwen. Chỉ chạy 1 lần!
QWEN = "Qwen/Qwen2.5-VL-7B-Instruct"
sh("run", "download.py", "--vlm", QWEN, "--skip-dataset", "--scorers", "")
p = subprocess.run([sys.executable, "launch.py", "blind", "--model", QWEN], env=ENV, capture_output=True,
                   text=True, encoding="utf-8", errors="replace")
print(p.stdout[-3000:], p.stderr[-2000:])

In [ ]:
# Ô 10b — Tiến độ + so sánh mức dựa vào video của 2 mô hình (cùng 1.000 câu)
from evaluate import extract_option, mcnemar_exact, load
from subset import load_qids

sh("volume", "get", VOL, "status/blind__Qwen2.5-VL-7B-Instruct.json",
   "outputs/status/blind__Qwen2.5-VL-7B-Instruct.json", "--force", check=False)
with open("outputs/status/blind__Qwen2.5-VL-7B-Instruct.json", encoding="utf-8") as f:
    stq = json.load(f)
print("done:", stq["done"], "| error:", stq.get("error"))

if stq["done"]:
    name = "textonly_0f__Qwen2.5-VL-7B-Instruct__subset_1000.csv"
    sh("volume", "get", VOL, f"results/{name}", f"outputs/results/{name}", "--force")
    sub_ids = set(load_qids("outputs/subsets/subset_1000.txt"))
    runs = {  # (mô hình, cấu hình) -> file kết quả
        ("InternVL2-8B", "0 frame"): "textonly_0f__InternVL2-8B__subset_1000.csv",
        ("InternVL2-8B", "Uniform-32"): "uniform_32f__InternVL2-8B__subset_1000.csv",
        ("InternVL2-8B", "KTV+E-32"): "ktv-e_32f__InternVL2-8B__subset_1000.csv",
        ("Qwen2.5-VL-7B", "0 frame"): name,
        ("Qwen2.5-VL-7B", "Uniform-32"): "uniform_32f__Qwen2.5-VL-7B-Instruct__full.csv",
        ("Qwen2.5-VL-7B", "AKS-CLIP-32"): "aks-clip_32f__Qwen2.5-VL-7B-Instruct__full.csv",
    }
    D = {}
    for k, f in runs.items():
        d = load(f"outputs/results/{f}")
        D[k] = d[d.Question_id.isin(sub_ids)].set_index("Question_id").sort_index()
    rows = []
    for (m, cfg), d in D.items():
        blind = D[(m, "0 frame")]
        ids = d.index.intersection(blind.index)
        diff = d.loc[ids, "correct"] - blind.loc[ids, "correct"]
        b, c = int((diff == 1).sum()), int((diff == -1).sum())
        rows.append({"model": m, "cấu hình": cfg, "n": len(ids), "Overall": 100 * d.loc[ids, "correct"].mean(),
                     "hơn 0 frame": 100 * diff.mean(), "p (McNemar)": mcnemar_exact(b, c),
                     "% giống 0 frame": 100 * (d.loc[ids, "pred"] == blind.loc[ids, "pred"]).mean()})
    display(pd.DataFrame(rows).round(3))

## Ô 11 — LLaVA-NeXT-Video-7B (Video-LLM chuyên video) @ 8 frame (stage `llava8`)

Kiểm tra kết luận "mô hình dựa ít vào video" với một mô hình **chuyên cho video** (Table 3: 38,6 @ 32f, cao hơn InternVL2-8B).
Trên subset_1000: **0 frame**, **Uniform-8**, **AKS+E-8** (t1 = 0.3, lấy từ stage f8), **KTV+E-8**.

- Dùng lại đúng file chọn frame 8f của stage f8 → chỉ đổi mô hình trả lời.
- Khác InternVL2: vLLM nhận frame dưới dạng **một video** (mảng T×H×W×3) với prompt `USER: <video>
... ASSISTANT:`;
  ngữ cảnh tối đa 4.096 token (8 frame × 144 token).
- Chi phí ước tính ~$3–4.

In [ ]:
# Ô 11a — Phát lệnh llava8 (deploy + spawn). Chỉ chạy 1 lần! (đã tải model: download.py --vlm llava-hf/LLaVA-NeXT-Video-7B-hf)
p = subprocess.run([sys.executable, "launch.py", "llava8"], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:], p.stderr[-2000:])

In [ ]:
# Ô 11b — Tiến độ + so sánh: LLaVA-NeXT-Video @8f vs InternVL2-8B @8f (cùng 1.000 câu, cùng frame)
from evaluate import mcnemar_exact, load

LL = "LLaVA-NeXT-Video-7B-hf"
sh("volume", "get", VOL, f"status/llava8__{LL}.json", f"outputs/status/llava8__{LL}.json", "--force", check=False)
with open(f"outputs/status/llava8__{LL}.json", encoding="utf-8") as f:
    stl = json.load(f)
print("done:", stl["done"], "| error:", stl.get("error"))
display(pd.DataFrame(stl["steps"]))

if stl["done"]:
    tags = ["textonly_0f", "uniform_8f", "aks-clip-t0.3-e_8f", "ktv-e_8f"]
    for t in tags:
        sh("volume", "get", VOL, f"results/{t}__{LL}__subset_1000.csv",
           f"outputs/results/{t}__{LL}__subset_1000.csv", "--force")
    subprocess.run([sys.executable, "evaluate.py"], env=ENV, capture_output=True)

    rows = []
    for m in (LL, MODEL_SHORT):
        D = {t: load(f"outputs/results/{t}__{m}__subset_1000.csv").set_index("Question_id").sort_index()
             for t in tags}
        b0, u = D["textonly_0f"], D["uniform_8f"]
        for t, d in D.items():
            i = d.index.intersection(b0.index)
            x0 = (d.loc[i, "correct"] - b0.loc[i, "correct"])
            xu = (d.loc[i, "correct"] - u.loc[i, "correct"])
            rows.append({"model": m, "tag": t, "Overall": 100 * d.loc[i, "correct"].mean(),
                         "unparsed": int(d["pred"].isna().sum()),
                         "hơn 0 frame": 100 * x0.mean(),
                         "p vs 0f": mcnemar_exact(int((x0 == 1).sum()), int((x0 == -1).sum())),
                         "hơn Uniform-8": 100 * xu.mean(),
                         "p vs Uniform": mcnemar_exact(int((xu == 1).sum()), int((xu == -1).sum())),
                         "% giống 0 frame": 100 * (d.loc[i, "pred"] == b0.loc[i, "pred"]).mean()})
    display(pd.DataFrame(rows).round(3))

## Ô 12 — MDP3+E-32 (ICCV 2025) trên InternVL2-8B (stage `mdp3`)

Port từ code chính thức `github.com/sunh-23/MDP3` (clone ở `third_party/MDP3`): SigLIP so400m cho frame + câu hỏi,
kernel Gaussian đa tỉ lệ, DPP có điều kiện câu hỏi (λ = 0,2), chia đoạn 32 frame + quy hoạch động phân bổ frame.
Bước chọn tham lam được viết lại dạng vector hóa — **kết quả giống hệt bản gốc** (kiểm tra 12/12 ca), nhanh hơn ~15 lần.
Ứng viên: tập chung với AKS/KTV (1 fps, tối thiểu 64), tối đa 128. Không dò tham số (dùng mặc định của code gốc).
So với Uniform-32 / AKS+E-32 / KTV+E-32 của lượt `night`. Chi phí ~$2–3.

In [ ]:
# Ô 12a — Phát lệnh mdp3 (deploy + spawn). Chỉ chạy 1 lần! (cần: download.py --vlm google/siglip-so400m-patch14-384)
p = subprocess.run([sys.executable, "launch.py", "mdp3"], env=ENV, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:], p.stderr[-2000:])

In [ ]:
# Ô 12b — Tiến độ + so sánh MDP3+E-32 với các phương pháp 32 frame (InternVL2-8B, subset_1000)
from evaluate import mcnemar_exact, load

sh("volume", "get", VOL, "status/mdp3.json", "outputs/status/mdp3.json", "--force", check=False)
with open("outputs/status/mdp3.json", encoding="utf-8") as f:
    stm = json.load(f)
print("done:", stm["done"], "| error:", stm.get("error"))
display(pd.DataFrame(stm["steps"]))

if stm["done"]:
    name = f"mdp3-e_32f__{MODEL_SHORT}__subset_1000.csv"
    sh("volume", "get", VOL, f"results/{name}", f"outputs/results/{name}", "--force")
    sh("volume", "get", VOL, "selections/mdp3-e_32f.timing.json", "outputs/selections/mdp3-e_32f.timing.json", "--force")
    subprocess.run([sys.executable, "evaluate.py"], env=ENV, capture_output=True)
    tags = {"0 frame": "textonly_0f", "Uniform-32": "uniform_32f", "AKS+E-32": "aks-clip-t0.2-e_32f",
            "KTV+E-32": "ktv-e_32f", "MDP3+E-32": "mdp3-e_32f"}
    D = {k: load(f"outputs/results/{t}__{MODEL_SHORT}__subset_1000.csv").set_index("Question_id").sort_index()
         for k, t in tags.items()}
    rows = []
    for k, d in D.items():
        r = {"phương pháp": k, "Overall": 100 * d.correct.mean()}
        for ref in ("0 frame", "Uniform-32", "KTV+E-32"):
            if k != ref:
                x = d.correct - D[ref].correct
                r[f"vs {ref}"] = 100 * x.mean()
                r[f"p vs {ref}"] = mcnemar_exact(int((x == 1).sum()), int((x == -1).sum()))
        for g in ("Recall", "Perception", "Reasoning", "Navigation"):
            r[g] = 100 * d[d.group == g].correct.mean()
        rows.append(r)
    display(pd.DataFrame(rows).round(3))